In [4]:
# Standard libraries
import os
from typing import List, Tuple

# Data manipulation and numerical computation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn utilities
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif, mutual_info_regression, f_classif, f_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Other utilities
from scipy.stats import randint, uniform
import joblib

# Standard libraries
from typing import List, Tuple

# Data manipulation and numerical computation
import numpy as np
import pandas as pd

# Scikit-learn utilities
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold, f_classif, f_regression

In [5]:
path = "../../data/raw/"
dfs = {}

# read all dataframes and keep them in dfs
for parquet in os.listdir(path):  # List all files in the directory
    if parquet.endswith(".parquet"):
        name = parquet.split(".parquet")[0]  # Get name without extension
        dfs[name] = pd.read_parquet(os.path.join(path, parquet))  # Use os.path.join for paths
        print(f"Loaded {name} with shape {dfs[name].shape}")

Loaded 01_DiatomInventories_GTstudentproject_B with shape (1643872, 8)
Loaded 02_InfoSites_GTstudentproject_B with shape (8404, 11)
Loaded 03_IBD_GTstudentproject_test with shape (5063, 2)
Loaded 03_IBD_GTstudentproject_train with shape (43783, 4)
Loaded 04_PressureStatus_GTstudentproject_B with shape (49231, 30)
Loaded 05_EnvParamMeans_GTstudentproject_B with shape (3763903, 8)
Loaded 06_ListEnvParam_GNNprojectGT_B with shape (192, 14)
Loaded 07_TaxaCode_GTstudentproject_B with shape (2292, 2)


In [6]:
taxones = dfs[list(dfs.keys())[0]] 

In [7]:
pressure = dfs[list(dfs.keys())[4]]

In [8]:
epm = dfs[list(dfs.keys())[5]]

In [9]:
path = "../../data/processed/"
dfs_processed = {}

# read all dataframes and keep them in dfs
for parquet in os.listdir(path):  # List all files in the directory
    if parquet.endswith(".parquet"):
        name = parquet.split(".parquet")[0]  # Get name without extension
        dfs_processed[name] = pd.read_parquet(os.path.join(path, parquet))  # Use os.path.join for paths
        print(f"Loaded {name} with shape {dfs_processed[name].shape}")

Loaded 03_CLEAN_COMPLETE_DF with shape (49863, 174)
Loaded 03_CLEAN_COMPLETE_DF_02 with shape (49231, 623)
Loaded 03_COMPLETE_TEST with shape (43568, 3)
Loaded 03_COMPLETE_TRAIN with shape (49441, 2332)
Loaded 03_COMPLETE_TRAIN_2 with shape (49231, 2849)
Loaded clean_train with shape (43568, 4)
Loaded dep_codes with shape (49231, 12)
Loaded dep_test with shape (5063, 12)
Loaded taxones_pressure with shape (5663, 2333)
Loaded taxones_pressure_epm_predict with shape (5663, 2850)
Loaded taxones_pressure_epm_train with shape (43568, 2853)
Loaded taxones_pressure_predict with shape (5663, 2333)
Loaded taxones_pressure_train with shape (43568, 2336)


In [10]:
sites = dfs_processed[list(dfs_processed.keys())[6]]

In [11]:
train = dfs_processed[list(dfs_processed.keys())[5]]

In [12]:
test = dfs_processed[list(dfs_processed.keys())[7]]

In [13]:
test_codes = test['CodeDepartement'].unique()

In [14]:
regiones = sites['HERlvl1Code'].drop_duplicates().tolist()
len(regiones)

22

ejemplo antes de volverlo función

In [15]:
region = 21

df = sites[sites['HERlvl1Code']== region]

valid_codes = df ['SamplingOperations_code'].unique()

df1 = taxones[taxones['SamplingOperations_code'].isin(valid_codes)]

# 1) columnas que QUIERES mantener en la llave (todas menos: TaxonName y Abundance_nbcell;
#    y quitamos TaxonCode/Abundance_pm porque se usan para columnas/valores)
cols_key = [c for c in df1.columns 
            if c not in ['TaxonName', 'Abundance_nbcell', 'TaxonCode', 'Abundance_pm']]

# 2) tabla pivote:
#    - index = todas las columnas que quieres conservar (parte del key)
#    - columns = códigos de taxón
#    - values = Abundance_pm
#    - aggfunc='sum' por si hay duplicados del mismo taxón en la misma muestra
wide = (
    df1.pivot_table(index=cols_key,
                   columns='TaxonCode',
                   values='Abundance_pm',
                   aggfunc='sum')
      .reset_index()
)

# opcional: quitar el nombre del eje de columnas
wide.columns.name = None

df2 = wide

df2 = pd.merge(wide, pressure, on=['SamplingOperations_code', 'CodeSite_SamplingOperations','Date_SamplingOperation'],how = 'inner')

train_df = pd.merge(df2, train, on= 'SamplingOperations_code',how = 'inner')
test_df = df2[df2['SamplingOperations_code'].isin(test_codes)]


In [16]:
def drop_exact_duplicates(X: pd.DataFrame) -> Tuple[pd.DataFrame, List[str]]:
    """
    Drop exact duplicate columns from a DataFrame.

    This function identifies and removes columns in the DataFrame that are exact duplicates 
    of other columns. Duplicate columns are those that have identical values across all rows.

    Parameters:
    ----------
    X : pd.DataFrame
        The input DataFrame containing the features.

    Returns:
    -------
    Tuple[pd.DataFrame, List[str]]
        - A DataFrame with duplicate columns removed.
        - A list of the names of the dropped duplicate columns.

    Notes:
    -----
    - The function computes a hash signature for each column to efficiently identify duplicates.
    - If two columns have the same hash and their values are identical, one of them is dropped.

    Example Usage:
    --------------
    X, dropped_dup_cols = drop_exact_duplicates(X)
    print(f"Dropped exact duplicate columns: {dropped_dup_cols}")
    """
    sig = X.apply(lambda s: pd.util.hash_pandas_object(s, index=False).sum())
    seen = {}
    dup = []
    for c, h in sig.items():
        if h in seen and X[c].equals(X[seen[h]]):
            dup.append(c)
        else:
            seen[h] = c
    return X.drop(columns=dup), dup

def drop_high_missing(X: pd.DataFrame, thresh: float = 0.40) -> Tuple[pd.DataFrame, List[str]]:
    """
    Drop columns with a high proportion of missing values from a DataFrame.

    This function identifies columns in the DataFrame where the proportion of missing values 
    exceeds a specified threshold and removes them. Missing values are identified as NaN 
    or the string "Unassessed", which is replaced with NaN before processing.

    Parameters:
    ----------
    X : pd.DataFrame
        The input DataFrame containing the features.
    thresh : float, optional
        The proportion threshold above which a column is considered to have high missing values 
        and is dropped. Default is 0.40 (40%).

    Returns:
    -------
    Tuple[pd.DataFrame, List[str]]
        - A DataFrame with high-missing columns removed.
        - A list of the names of the dropped columns.

    Notes:
    -----
    - The function replaces the string "Unassessed" with NaN before calculating the proportion 
      of missing values.
    - Columns with a proportion of missing values greater than `thresh` are dropped.

    Example Usage:
    --------------
    X, dropped_cols = drop_high_missing(X, thresh=0.40)
    print(f"Dropped columns with >40% missing: {dropped_cols}")
    """
    # Replace "Unassessed" with NaN
    X = X.replace("Unassessed", np.nan)

    # Identify columns with a high proportion of missing values
    to_drop = X.columns[X.isna().mean() > thresh].tolist()

    # Drop the identified columns
    X2 = X.drop(columns=to_drop)

    return X2, to_drop

def drop_quasi_constant_cat(
    X: pd.DataFrame, 
    cat_cols: List[str], 
    p: float = 0.99
) -> Tuple[pd.DataFrame, List[str]]:
    """
    Drop quasi-constant categorical columns from a DataFrame.

    This function identifies categorical columns where the most frequent value 
    accounts for at least `p` proportion of the data and removes them from the DataFrame. 
    Quasi-constant columns are those that provide little variability and are unlikely 
    to be useful for modeling.

    Parameters:
    ----------
    X : pd.DataFrame
        The input DataFrame containing the features.
    cat_cols : List[str]
        A list of categorical column names to consider for quasi-constant checking.
    p : float, optional
        The proportion threshold above which a column is considered quasi-constant.
        Default is 0.99.

    Returns:
    -------
    Tuple[pd.DataFrame, List[str]]
        - A DataFrame with quasi-constant categorical columns removed.
        - A list of the names of the dropped quasi-constant categorical columns.

    Notes:
    -----
    - The function ensures that only columns present in the DataFrame are processed.
    - Missing values are included in the value counts when determining the most frequent value.

    Example Usage:
    --------------
    cat_cols = X.select_dtypes(exclude=['number']).columns
    X, dropped_cat_cols = drop_quasi_constant_cat(X, cat_cols.tolist(), p=0.99)
    print(f"Dropped quasi-constant categorical columns: {dropped_cat_cols}")
    """
    dropped = []
    for c in cat_cols:
        # Calculate the normalized value counts (including NaNs)
        vc = X[c].value_counts(normalize=True, dropna=False)
        
        # Check if the most frequent value exceeds the threshold
        if len(vc) and vc.iloc[0] >= p:
            dropped.append(c)
    
    # Drop the identified quasi-constant columns
    X2 = X.drop(columns=dropped)
    return X2, dropped

def drop_high_cardinality_cat(
    X: pd.DataFrame, 
    cat_cols: List[str], 
    max_unique: int = 100, 
    ratio: float = 0.50
) -> Tuple[pd.DataFrame, List[str]]:
    """
    Drop high-cardinality categorical columns from a DataFrame.

    This function identifies categorical columns with a high number of unique values 
    (cardinality) and removes them from the DataFrame. High-cardinality columns can 
    increase the complexity of the model and may not provide significant value.

    Parameters:
    ----------
    X : pd.DataFrame
        The input DataFrame containing the features.
    cat_cols : List[str]
        A list of categorical column names to consider for cardinality checking.
    max_unique : int, optional
        The maximum number of unique values allowed in a categorical column. 
        Default is 100.
    ratio : float, optional
        The maximum ratio of unique values to the total number of rows in the DataFrame.
        Default is 0.50.

    Returns:
    -------
    Tuple[pd.DataFrame, List[str]]
        - A DataFrame with high-cardinality categorical columns removed.
        - A list of the names of the dropped high-cardinality categorical columns.

    Notes:
    -----
    - A column is considered high-cardinality if the number of unique values exceeds 
      `max_unique` or if the ratio of unique values to the total number of rows exceeds `ratio`.
    - The function ensures that only columns present in the DataFrame are processed.

    Example Usage:
    --------------
    cat_cols = X.select_dtypes(exclude=['number']).columns
    X, dropped_high_card_cols = drop_high_cardinality_cat(X, cat_cols.tolist(), max_unique=100, ratio=0.50)
    print(f"Dropped high-cardinality categorical columns: {dropped_high_card_cols}")
    """
    n = len(X)
    lim = min(max_unique, int(ratio * n))
    dropped = [c for c in cat_cols if c in X.columns and X[c].nunique(dropna=False) > lim]
    X2 = X.drop(columns=dropped)
    return X2, dropped

def drop_quasi_constant_num(
    X: pd.DataFrame, num_cols: List[str], thresh: float = 1e-5
) -> Tuple[pd.DataFrame, List[str]]:
    """
    Drop quasi-constant numeric columns from a DataFrame.

    This function identifies numeric columns with very low variance (below a specified threshold)
    and removes them from the DataFrame. Quasi-constant columns are those that have almost the same
    value across all rows, which makes them uninformative for modeling.

    Parameters:
    ----------
    X : pd.DataFrame
        The input DataFrame containing the features.
    num_cols : List[str]
        A list of numeric column names to consider for variance checking.
    thresh : float, optional
        The variance threshold below which columns are considered quasi-constant.
        Default is 1e-5.

    Returns:
    -------
    Tuple[pd.DataFrame, List[str]]
        - A DataFrame with quasi-constant numeric columns removed.
        - A list of the names of the dropped quasi-constant numeric columns.

    Notes:
    -----
    - Missing values in the numeric columns are imputed using the median before calculating variance.
    - The VarianceThreshold from sklearn is used to identify quasi-constant columns.
    """
    if not num_cols:
        return X, []

    # Impute missing values with the median
    imp = SimpleImputer(strategy="median")
    Xn = pd.DataFrame(imp.fit_transform(X[num_cols]), columns=num_cols, index=X.index)

    # Apply VarianceThreshold to identify columns with low variance
    vt = VarianceThreshold(threshold=thresh)
    vt.fit(Xn)

    # Identify kept and dropped columns
    kept = [c for c, k in zip(num_cols, vt.get_support()) if k]
    dropped = [c for c in num_cols if c not in kept]

    # Drop the quasi-constant columns from the original DataFrame
    X2 = X.drop(columns=dropped)
    return X2, dropped

def prefilter_num_univariate(
    X: pd.DataFrame, y: pd.Series, num_cols: List[str], k: int = 300
) -> Tuple[pd.DataFrame, List[str], pd.Series]:
    """
    Perform a univariate feature selection for numeric columns based on their relationship with the target variable.

    This function selects the top `k` numeric features that have the highest scores in a univariate statistical test 
    (ANOVA F-value for classification or F-statistic for regression) with respect to the target variable `y`.

    Parameters:
    ----------
    X : pd.DataFrame
        The input dataframe containing features.
    y : pd.Series
        The target variable.
    num_cols : List[str]
        A list of numeric column names to consider for feature selection.
    k : int, optional
        The number of top features to keep, by default 300.

    Returns:
    -------
    Tuple[pd.DataFrame, List[str], pd.Series]
        - A dataframe containing the top `k` numeric features.
        - A list of the names of the selected top `k` numeric features.
        - A pandas Series containing the scores of all numeric features, sorted in descending order.

    Notes:
    -----
    - If the target variable `y` is numeric and has more than 20 unique values, the function uses `f_regression`.
      Otherwise, it uses `f_classif`.
    - Missing values in the numeric columns are imputed using the median before calculating the scores.
    """
    # Filter numeric columns that exist in the dataframe
    keep = [c for c in num_cols if c in X.columns]
    if not keep:
        return X, [], pd.Series(dtype=float)

    # Impute missing values with the median
    Xi = pd.DataFrame(
        SimpleImputer(strategy="median").fit_transform(X[keep]),
        columns=keep, index=X.index
    )

    # Select the appropriate statistical test based on the target variable type
    if np.issubdtype(y.dtype, np.number) and y.nunique() > 20:
        scores, _ = f_regression(Xi, y.to_numpy())
    else:
        scores, _ = f_classif(Xi, y.to_numpy())

    # Create a pandas Series of scores, sort them in descending order
    s = pd.Series(scores, index=keep).fillna(0.0).sort_values(ascending=False)

    # Select the top `k` features
    top = s.index[: min(k, len(s))].tolist()

    # Return the top features, their names, and the scores
    return pd.concat([Xi[top]], axis=1), top, s

def drop_zeros(X: pd.DataFrame, thresh: float = 0.95) -> Tuple[pd.DataFrame, List[str]]:
    """
    Drop columns with a high proportion of zero values from a DataFrame.

    This function identifies columns in the DataFrame where the proportion of zero values 
    exceeds a specified threshold and removes them. Columns with a high proportion of zeros 
    are often uninformative and can be excluded from further analysis.

    Parameters:
    ----------
    X : pd.DataFrame
        The input DataFrame containing the features.
    thresh : float, optional
        The proportion threshold above which a column is considered to have high zero values 
        and is dropped. Default is 0.95 (95%).

    Returns:
    -------
    Tuple[pd.DataFrame, List[str]]
        - A DataFrame with high-zero columns removed.
        - A list of the names of the dropped columns.

    Notes:
    -----
    - The function calculates the proportion of zero values in each column.
    - Columns with a proportion of zero values greater than `thresh` are dropped.

    Example Usage:
    --------------
    Xnum, dropped_cols = drop_zeros(Xnum, thresh=0.95)
    print(f"Dropped columns with >95% zeros: {dropped_cols}")
    """
    # Identify columns with a high proportion of zero values
    to_drop = X.columns[(X == 0).mean() > thresh].tolist()

    # Drop the identified columns
    X = X.drop(columns=to_drop)

    return X, to_drop


In [17]:
def clean_up(train,test):
    train.set_index('SamplingOperations_code')
    Xtr = train.drop(columns=(['IBD', 'IBD_EQR', 'IBD_EQR_Status'])).set_index('SamplingOperations_code')
    Y = train[['SamplingOperations_code', 'IBD', 'IBD_EQR', 'IBD_EQR_Status']].set_index('SamplingOperations_code')
    y = train[['IBD']]
    Xte=test.set_index('SamplingOperations_code')
    
    X = pd.concat([Xtr, Xte], axis=0)
    COMPLETE_X = X.copy()
    X = X.drop(columns=['Date_SamplingOperation'])

    """En este espacio es dodne podemos hacer limpieza de datos """
    X, dropped_dup_cols = drop_exact_duplicates(X)
    print(f"Dropped exact duplicate columns: {dropped_dup_cols}")

    X, dropped_cols = drop_high_missing(X, thresh=0.95)
    print(f"Dropped columns with >95% missing: {dropped_cols}")
    print(len(dropped_cols))

    cat_cols = X.select_dtypes(exclude=['number']).columns
    X, dropped_cat_cols = drop_quasi_constant_cat(X, cat_cols.tolist(), p=0.99)
    print(f"Dropped quasi-constant categorical columns: {dropped_cat_cols}")

    cat_cols = X.select_dtypes(exclude=['number']).columns
    X, dropped_high_card_cols = drop_high_cardinality_cat(X, cat_cols.tolist(), max_unique=100, ratio=0.50)
    print(f"Dropped high-cardinality categorical columns: {dropped_high_card_cols}")

    num_cols = X.select_dtypes(include=['number']).columns  # Select numeric columns
    X, dropped_num_cols = drop_quasi_constant_num(X, num_cols.tolist(), thresh=1e-5)
    print(f"Dropped quasi-constant numeric columns: {dropped_num_cols}")
    
    COMPLETE_X[dropped_cols]

    uncommon_taxons = COMPLETE_X[dropped_cols].sum(axis=1)
    uncommon_taxons = uncommon_taxons.rename("Uncommon_Taxons")

    df3 = pd.concat([X, uncommon_taxons], axis=1)

    df3 = df3.join(Y, how='left')

    

    return df3 

In [18]:
def get_region_sites_and_taxa(region, sites, taxones):
    """
    Regresa los dos DF que armaste a mano:
    - df: subset de sites para la región
    - df1: subset de taxones filtrado por los SamplingOperations_code válidos de df
    """
    df = sites[sites['HERlvl1Code'] == region].copy()
    valid_codes = df['SamplingOperations_code'].unique()
    df1 = taxones[taxones['SamplingOperations_code'].isin(valid_codes)].copy()
    return  df1


In [19]:
def get_region_sites_and_epm(region, sites, epm):
    """
    Regresa los dos DF que armaste a mano:
    - df: subset de sites para la región
    - df1: subset de taxones filtrado por los SamplingOperations_code válidos de df
    """
    df = sites[sites['HERlvl1Code'] == region].copy()
    valid_codes = df['SamplingOperations_code'].unique()
    dfe = epm[epm['SamplingOperations_code'].isin(valid_codes)].copy()
    return  dfe

In [20]:
def pivot_taxa_abundance_pm(df1):
    """
    Hace el pivote exactamente como en tu código:
    - index: todas las columnas salvo TaxonName, Abundance_nbcell, TaxonCode, Abundance_pm
    - columns: TaxonCode
    - values: Abundance_pm
    - aggfunc: sum
    """
    # columnas que conservas en la llave
    drop_cols = ['TaxonName', 'Abundance_nbcell', 'TaxonCode', 'Abundance_pm']
    cols_key = [c for c in df1.columns if c not in drop_cols]

    wide = (
        df1.pivot_table(index=cols_key,
                        columns='TaxonCode',
                        values='Abundance_pm',
                        aggfunc='sum')
           .reset_index()
    )
    wide.columns.name = None
    return wide


In [21]:
def pivot_pressure_means(dfe, agg='first'):
    """
    Pivotea un df de presión para obtener columnas por código de parámetro y
    valores = Mean1Y / Mean180Days / Mean90Days.

    Parámetros
    ----------
    df : DataFrame
        Debe contener: 
        ['CodeSite_SamplingOperations', 'SamplingOperations_code', 
         'Parametre_code', 'Mean1Y', 'Mean180Days', 'Mean90Days'].
    agg : str or callable, default 'first'
        Función de agregación cuando hay duplicados por (keys, Parametre_code).
        Ejemplos: 'first', 'mean', 'max', np.mean, etc.
    drop_code_site : bool, default True
        Si True, elimina la columna 'CodeSite_SamplingOperations' al final.

    Return
    ------
    wide2 : DataFrame
        DataFrame ancho con columnas como:
        Mean1Y_XXXX, Mean180Days_XXXX, Mean90Days_XXXX (XXXX = Parametre_code),
        e índice reseteado.
    """
    # Claves del índice
    keys = ['CodeSite_SamplingOperations', 'SamplingOperations_code']

    # Pivot con columnas multiíndice (stat, Parametre_code)
    wide = (
        dfe.pivot_table(
            index=keys,
            columns='Parametre_code',
            values=['Mean1Y', 'Mean180Days', 'Mean90Days'],
            aggfunc=agg
        )
        .sort_index(axis=1)  # orden estable
    )

    # Aplana nombres de columnas -> Mean1Y_1319, Mean180Days_1319, ...
    wide.columns = [f"{stat}_{code}" for stat, code in wide.columns]

    # Resetea índice para volver a columnas normales
    wide2 = wide.reset_index()


    return wide2


In [22]:
def build_region_train_test(region, sites, taxones, pressure, train, test_codes, fillna_with=0):
    """
    Flujo completo por región:
    1) Filtra sites y taxones (df, df1)
    2) Pivotea abundancias permil por TaxonCode (wide)
    3) Une con pressure (df2) usando las mismas llaves que tú pusiste
    4) Saca train_df (inner contra 'train' por SamplingOperations_code)
    5) Saca test_df filtrando df2 por test_codes

    Parámetros:
      - region: int/str con el código de región (HERlvl1Code)
      - sites, taxones, pressure, train: DataFrames existentes
      - test_codes: iterable de SamplingOperations_code para test
      - fillna_with: si quieres rellenar NaN post-pivote (0 por default). Usa None para no rellenar.

    Regresa:
      - train_df, test_df
    """
    # 1) df y df1
    df1 = get_region_sites_and_taxa(region, sites, taxones)

    # Si no hay datos para la región, devuelve vacíos con mismas columnas.
    if df1.empty:
        empty_train = pd.DataFrame(columns=['SamplingOperations_code'])
        empty_test  = pd.DataFrame(columns=['SamplingOperations_code'])
        return empty_train, empty_test

    # 2) pivote
    wide = pivot_taxa_abundance_pm(df1)

    # 3) merge con pressure (exactamente tus llaves)
    df2 = pd.merge(
        wide,
        pressure,
        on=['SamplingOperations_code', 'CodeSite_SamplingOperations', 'Date_SamplingOperation'],
        how='inner'
    )

   
    df3 = pd.merge(df2, train, on='SamplingOperations_code', how='left')    
    # 5) test_df (filtrado por test_codes)
    col = 'IBD'   # columna en dfB que checas

    mask = df3[col].isna()          # True si NaN

    train_df = df3.loc[~mask]
    test_df = df3.loc[mask]  
    test_df=test_df.drop(columns=(['IBD', 'IBD_EQR', 'IBD_EQR_Status']))

    cleandf = clean_up(train= train_df,test=test_df)

    return cleandf


In [30]:
# Ejemplo de uso:
region = 21
cleandf= build_region_train_test(
    region=region,
    sites=sites,
    taxones=taxones,
    pressure=pressure,
    train=train,
    test_codes=test_codes,   # asegúrate de tenerlo definido
    fillna_with=0            # pon None si no quieres rellenar
)


Dropped exact duplicate columns: ['Pinrh02']
Dropped columns with >95% missing: ['Achaa01', 'Achac01', 'Achaf01', 'Achaf02', 'Achal01', 'Achan01', 'Achba01', 'Achbi02', 'Achca03', 'Achca04', 'Achco01', 'Achco02', 'Achcr01', 'Achde02', 'Achde03', 'Achdi01', 'Achdi02', 'Achdr01', 'Achel01', 'Achen01', 'Achex01', 'Achex02', 'Achfr01', 'Achfu01', 'Achge01', 'Achgr02', 'Achgr03', 'Achhe01', 'Achho01', 'Achhu01', 'Achja02', 'Achjo01', 'Achko01', 'Achku01', 'Achla03', 'Achla04', 'Achla06', 'Achle02', 'Achli01', 'Achli02', 'Achli03', 'Achlu02', 'Achma01', 'Achmi03', 'Achna02', 'Achne01', 'Achno01', 'Achpa01', 'Achpe02', 'Achpf01', 'Achps04', 'Achpu01', 'Achpy01', 'Achre01', 'Achro01', 'Achro02', 'Achru01', 'Achsa01', 'Achse02', 'Achsi01', 'Achst01', 'Achst03', 'Achsu06', 'Achsu07', 'Achte01', 'Achtr02', 'Achtr03', 'Achzh01', 'Achzi01', 'Actde01', 'Actno01', 'Adlba01', 'Adlbr01', 'Adlbr02', 'Adlmu02', 'Adlpa01', 'Adlsu01', 'Ampat01', 'Ampei01', 'Ampma01', 'Ampme01', 'Ampmi02', 'Ampmo01', 'Ampov

In [31]:
cleandf

,TotalAbundance_SamplingOperation,Achat02,Achca02,Achda01,Achda02,Acheu01,Achho03,Achkr02,Achla02,Achmi02,...,OrganicMicropollutants_Status1Y,OrganicMicropollutants_Status180D,OrganicMicropollutants_Status90D,MineralMicropollutants_Status1Y,MineralMicropollutants_Status180D,MineralMicropollutants_Status90D,Uncommon_Taxons,IBD,IBD_EQR,IBD_EQR_Status
SamplingOperations_code,,,,,,,,,,,,,,,,,,,,,
S03024230_20140707,410,NaN,NaN,NaN,NaN,NaN,NaN,NaN,36.585366,68.292683,...,NaN,NaN,NaN,NaN,NaN,NaN,112.195122,17.6,0.900000,Good
S03024230_20190625,402,NaN,2.487562,NaN,NaN,NaN,4.975124,NaN,27.363184,NaN,...,Good,Good,Good,Moderate,Moderate,Moderate,116.915423,20.0,1.000000,High
S03024245_20130930,413,NaN,NaN,NaN,NaN,NaN,NaN,NaN,67.796610,234.866828,...,Moderate,Moderate,Moderate,Poor,Bad,Poor,50.847458,16.0,0.785714,Good
S03024245_20150611,423,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.364066,151.300236,...,Moderate,Moderate,Moderate,Moderate,Moderate,Moderate,18.912530,17.1,0.864286,Good
S03024245_20160726,410,NaN,NaN,NaN,NaN,NaN,48.780488,NaN,7.317073,51.219512,...,Moderate,Moderate,Moderate,Moderate,Moderate,Moderate,24.390244,18.9,0.992857,High
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
S05069975_20180806,406,NaN,19.704433,4.926108,NaN,NaN,NaN,NaN,22.167488,150.246305,...,NaN,NaN,NaN,NaN,NaN,NaN,68.965517,NaN,NaN,NaN
S05071300_20130702,420,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.523810,154.761905,...,Good,Good,Good,Moderate,Moderate,Good,69.047619,NaN,NaN,NaN
S05071300_20160819,400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,115.000000,...,Moderate,Good,Good,Moderate,NaN,NaN,17.500000,NaN,NaN,NaN


In [24]:
KEY = "SamplingOperations_code"

def save_with_key(df, path, key=KEY):
    # si la llave está como índice, bájala a columna
    if key not in df.columns and df.index.name == key:
        df = df.reset_index()

    # checks de sanidad
    assert key in df.columns, f"Falta la llave '{key}'"
    assert df[key].notna().all(), f"Hay NaNs en la llave '{key}'"
    assert df[key].is_unique, f"La llave '{key}' no es única"

    df.to_parquet(path, index=False)
    print(f"✔ Guardado con llave como columna -> {path}")

In [25]:
import os


# Definir el path base
base_path = r"dfs_taxon_p"

# Crear la carpeta si no existe
"""os.makedirs(base_path, exist_ok=True)

# Loop por regiones y guardar cada df
for region in regiones:
    cleandf = build_region_train_test(
        region=region,
        sites=sites,
        taxones=taxones,
        pressure=pressure,
        train=train,
        test_codes=test_codes,
        fillna_with=0
    )

    # Nombre del archivo, por ejemplo df_PACA.parquet
    file_name = f"df_{region}.parquet"
    file_path = os.path.join(base_path, file_name)



    # Guardar el DataFrame
    save_with_key(cleandf, file_path + file_name)
    print(f"Guardado: {file_path}")
"""




'os.makedirs(base_path, exist_ok=True)\n\n# Loop por regiones y guardar cada df\nfor region in regiones:\n    cleandf = build_region_train_test(\n        region=region,\n        sites=sites,\n        taxones=taxones,\n        pressure=pressure,\n        train=train,\n        test_codes=test_codes,\n        fillna_with=0\n    )\n\n    # Nombre del archivo, por ejemplo df_PACA.parquet\n    file_name = f"df_{region}.parquet"\n    file_path = os.path.join(base_path, file_name)\n\n\n\n    # Guardar el DataFrame\n    save_with_key(cleandf, file_path + file_name)\n    print(f"Guardado: {file_path}")\n'

In [40]:
# ===== helper: componentes conexas =====
def _connected_components_from_adj(adj_df: pd.DataFrame) -> list[list[str]]:
    nodes = adj_df.index.tolist()
    A = adj_df.values.astype(bool)
    visited = set()
    comps = []
    for i in range(len(nodes)):
        if nodes[i] in visited:
            continue
        stack = [i]
        comp = set()
        while stack:
            u = stack.pop()
            if u in comp:
                continue
            comp.add(u)
            neigh = np.where(A[u])[0]
            for w in neigh:
                if w not in comp:
                    stack.append(w)
        for j in comp:
            visited.add(nodes[j])
        comps.append([nodes[j] for j in comp])
    return comps

# ===== helper: correlación robusta numérica =====
def _build_robust_corr(
    df: pd.DataFrame,
    num_cols=None,
    method="spearman",
    min_pairs=2000,
    min_pair_coverage=0.10,
    min_col_coverage=0.10,
    const_std_tol=1e-12
):
    if num_cols is None:
        num_cols = df.select_dtypes(include=["number"]).columns.tolist()

    num_df = df[num_cols].copy().replace([np.inf, -np.inf], np.nan)
    N = len(num_df)

    # filtra columnas con poca cobertura o constantes
    col_cov = num_df.notna().mean(0)
    keep = col_cov[col_cov >= min_col_coverage].index.tolist()
    num_df = num_df[keep]
    if num_df.shape[1] == 0:
        return pd.DataFrame(), keep

    stds = num_df.std(skipna=True)
    nunq = num_df.nunique(dropna=True)
    const_cols = set(stds[stds <= const_std_tol].index) | set(nunq[nunq <= 1].index)
    keep = [c for c in keep if c not in const_cols]
    num_df = num_df[keep]
    if num_df.shape[1] == 0:
        return pd.DataFrame(), keep

    valid = num_df.notna().astype(int)
    pair_counts = pd.DataFrame(valid.T.values @ valid.values, index=keep, columns=keep)
    pair_cov = pair_counts / float(N)

    corr = num_df.corr(method=method)
    low_ev = (pair_counts < min_pairs) | (pair_cov < min_pair_coverage)
    corr = corr.mask(low_ev)
    return corr, keep

# ===== helper: eta² y Cramer's V corregido =====
from scipy.stats import chi2_contingency

def _eta2(y: pd.Series, x: pd.Series) -> float:
    df_ = pd.DataFrame({'y': y, 'x': x}).dropna()
    if df_.empty:
        return np.nan
    overall = df_['y'].mean()
    grp = df_.groupby('x')['y'].agg(['count','mean'])
    ssb = (grp['count'] * (grp['mean'] - overall)**2).sum()
    sst = ((df_['y'] - overall)**2).sum()
    return float(ssb/sst) if sst > 0 else np.nan

def _cramers_v(a: pd.Series, b: pd.Series) -> float:
    df_ = pd.DataFrame({'a': a, 'b': b}).dropna()
    n = len(df_)
    if n == 0:
        return np.nan
    tbl = pd.crosstab(df_['a'], df_['b'])
    r, k = tbl.shape
    if r < 2 or k < 2:
        return np.nan
    chi2, _, _, _ = chi2_contingency(tbl, correction=False)
    phi2 = chi2 / n
    phi2corr = max(0.0, phi2 - ((k - 1)*(r - 1))/max(n - 1, 1))
    rcorr = r - ((r - 1)**2)/max(n - 1, 1)
    kcorr = k - ((k - 1)**2)/max(n - 1, 1)
    denom = max(min(rcorr - 1, kcorr - 1), 1e-12)
    v = np.sqrt(phi2corr / denom)
    return float(np.clip(v, 0.0, 1.0))

# ====== LA FUNCIÓN QUE QUIERES: SOLO DEVUELVE el NUEVO DF ======
def cleaner_df_only(
    cleandf: pd.DataFrame,
    index_col='SamplingOperations_code',
    target_cols=('IBD','IBD_EQR','IBD_EQR_Status'),
    # umbrales numéricos
    num_corr_thr=0.95,
    num_corr_method='spearman',
    min_pairs=2000,
    min_pair_cov=0.10,
    min_col_cov=0.20,
    # umbrales categóricos
    cat_cov_min=0.20,
    cat_dom_thr=0.98,
    cat_max_levels=50,
    cat_cramers_thr=0.80
) -> pd.DataFrame:
    """
    Segundo clean: quita SOLO columnas redundantes.
    Mantiene formato:
      - Índice = index_col
      - Conserva columnas objetivo si existen
      - No devuelve listas de kept/dropped
    """
    df = cleandf.copy()

    # Asegura que index_col exista como columna (por si ya venía como índice)
    if index_col in df.index.names:
        df = df.reset_index()

    # Subconjunto con target disponible para medir asociación sin fuga de información
    has_y = df[target_cols[0]].notna() if target_cols[0] in df.columns else pd.Series(False, index=df.index)
    train_part = df.loc[has_y].copy()

    # ---------------- NUMÉRICAS ----------------
    feat_cols = [c for c in df.columns if c not in (list(target_cols) + [index_col])]
    num_cols = train_part[feat_cols].select_dtypes(include=['number']).columns.tolist()
    kept_num = num_cols
    if len(num_cols) > 1:
        corr, prelim_num = _build_robust_corr(
            train_part[num_cols],
            num_cols=num_cols,
            method=num_corr_method,
            min_pairs=min_pairs,
            min_pair_coverage=min_pair_cov,
            min_col_coverage=min_col_cov
        )
        if corr is not None and not corr.empty:
            # correlación con el objetivo (absoluta)
            xy = train_part[prelim_num + [target_cols[0]]].dropna()
            if not xy.empty:
                tcor = xy[prelim_num].corrwith(xy[target_cols[0]], method=num_corr_method).abs().to_dict()

                adj = corr.abs() >= num_corr_thr
                adj = adj.fillna(False)
                np.fill_diagonal(adj.values, False)
                comps = _connected_components_from_adj(adj)

                keep, drop = set(), set()
                for comp in comps:
                    if len(comp) == 1:
                        keep.add(comp[0]); continue
                    best = max(comp, key=lambda c: (tcor.get(c, np.nan),))
                    keep.add(best)
                    drop.update([c for c in comp if c != best])

                all_in_groups = set(sum(comps, []))
                isolated = set(prelim_num) - all_in_groups
                keep.update(isolated)
                kept_num = sorted(list(keep))
            else:
                kept_num = prelim_num  # sin pares válidos con y

    # ---------------- CATEGÓRICAS ----------------
    cat_cols = sorted(list(set(feat_cols) - set(num_cols)))
    kept_cat = cat_cols
    if len(cat_cols) > 0 and not train_part.empty:
        cat_df = train_part[cat_cols].replace([np.inf, -np.inf], np.nan)
        # métricas rápidas
        cov = cat_df.notna().mean(0)
        nun = cat_df.nunique(dropna=True)
        dom = cat_df.apply(lambda s: s.value_counts(dropna=True, normalize=True).iloc[0] if s.nunique(dropna=True) > 0 else np.nan)

        candidates = [c for c in cat_cols if (cov.get(c,0) >= cat_cov_min and nun.get(c,0) > 1)]
        if candidates:
            # eta² con target
            eta2_map = {c: _eta2(train_part[target_cols[0]], cat_df[c]) for c in candidates}
            # limitar cardinalidad
            levels = cat_df[candidates].nunique(dropna=True).to_dict()
            small = [c for c in candidates if levels.get(c, 0) <= cat_max_levels]

            if len(small) > 1:
                N = len(cat_df)
                notna_mat = cat_df[small].notna().astype(int)
                pair_counts = pd.DataFrame(notna_mat.T.values @ notna_mat.values, index=small, columns=small)
                pair_cov = pair_counts / float(N)

                V = pd.DataFrame(index=small, columns=small, dtype=float)
                for i, c1 in enumerate(small):
                    V.loc[c1, c1] = 1.0
                    for j in range(i+1, len(small)):
                        c2 = small[j]
                        if (pair_counts.loc[c1, c2] < min_pairs) or (pair_cov.loc[c1, c2] < min_pair_cov):
                            v = np.nan
                        else:
                            v = _cramers_v(cat_df[c1], cat_df[c2])
                        V.loc[c1, c2] = v
                        V.loc[c2, c1] = v

                adj = (V >= cat_cramers_thr).fillna(False)
                np.fill_diagonal(adj.values, False)
                comps = _connected_components_from_adj(adj)

                keep, drop = set(), set()
                for comp in comps:
                    if len(comp) == 1:
                        keep.add(comp[0]); continue
                    best = max(comp, key=lambda c: (eta2_map.get(c, np.nan), -levels.get(c, 10**9)))
                    keep.add(best)
                    drop.update([c for c in comp if c != best])

                all_in_groups = set(sum(comps, []))
                isolated = set(small) - all_in_groups
                keep.update(isolated)
                kept_cat = sorted(list(keep))
            else:
                kept_cat = small if small else candidates

    # ---------------- CONSTRUIR DF FINAL (MISMO FORMATO) ----------------
    keep_features = sorted(set(kept_num) | set(kept_cat))
    cols_final = [index_col] + keep_features + [c for c in target_cols if c in df.columns]
    cols_final = [c for c in cols_final if c in df.columns]  # por seguridad

    out = df[cols_final].copy()
    # índice requerido
    if index_col in out.columns:
        out = out.set_index(index_col)

    return out


In [41]:
def build_region_train_test_epm(region, sites, taxones, pressure, train, test_codes, fillna_with=0):
    """
    Flujo completo por región:
    1) Filtra sites y epm (df, df1)
    1.5) Filtra sites y taxones (df, dfe)
    2) Pivotea abundancias permil por TaxonCode (wide)
    3) Une con pressure (df2) usando las mismas llaves que tú pusiste
    4) Saca train_df (inner contra 'train' por SamplingOperations_code)
    5) Saca test_df filtrando df2 por test_codes

    Parámetros:
      - region: int/str con el código de región (HERlvl1Code)
      - sites, taxones, pressure, train: DataFrames existentes
      - test_codes: iterable de SamplingOperations_code para test
      - fillna_with: si quieres rellenar NaN post-pivote (0 por default). Usa None para no rellenar.

    Regresa:
      - train_df, test_df
    """
    # 1) df y df1
    df1 = get_region_sites_and_taxa(region, sites, taxones)
    
    dfe = get_region_sites_and_epm(region, sites, epm)

    # Si no hay datos para la región, devuelve vacíos con mismas columnas.
    if df1.empty:
        empty_train = pd.DataFrame(columns=['SamplingOperations_code'])
        empty_test  = pd.DataFrame(columns=['SamplingOperations_code'])
        return empty_train, empty_test

    # 2) pivote
    wide = pivot_taxa_abundance_pm(df1)
    wide2=pivot_pressure_means(dfe)

    # 3) merge con pressure (exactamente tus llaves)
    df2 = pd.merge(
        wide,
        pressure,
        on=['SamplingOperations_code', 'CodeSite_SamplingOperations', 'Date_SamplingOperation'],
        how='inner'
    )

    df2 = pd.merge(
        df2,
        wide2,
        on=['SamplingOperations_code'],
        how='inner'
    )

   
    df3 = pd.merge(df2, train, on='SamplingOperations_code', how='left')    
    # 5) test_df (filtrado por test_codes)
    col = 'IBD'   # columna en dfB que checas

    mask = df3[col].isna()          # True si NaN

    train_df = df3.loc[~mask]
    test_df = df3.loc[mask]  
    test_df=test_df.drop(columns=(['IBD', 'IBD_EQR', 'IBD_EQR_Status']))

    cleandf = clean_up(train= train_df,test=test_df)

    cleandf = cleaner_df_only(cleandf)

    return cleandf

In [42]:
# Ejemplo de uso:
region = 21
cleandf= build_region_train_test_epm(
    region=region,
    sites=sites,
    taxones=taxones,
    pressure=pressure,
    train=train,
    test_codes=test_codes,   # asegúrate de tenerlo definido
    fillna_with=0            # pon None si no quieres rellenar
)

Dropped exact duplicate columns: ['Pinrh02', 'Mean1Y_1770', 'Mean90Days_1647', 'Mean90Days_1648']
Dropped columns with >95% missing: ['Achaa01', 'Achac01', 'Achaf01', 'Achaf02', 'Achal01', 'Achan01', 'Achba01', 'Achbi02', 'Achca03', 'Achca04', 'Achco01', 'Achco02', 'Achcr01', 'Achde02', 'Achde03', 'Achdi01', 'Achdi02', 'Achdr01', 'Achel01', 'Achen01', 'Achex01', 'Achex02', 'Achfr01', 'Achfu01', 'Achge01', 'Achgr02', 'Achgr03', 'Achhe01', 'Achho01', 'Achhu01', 'Achja02', 'Achjo01', 'Achko01', 'Achku01', 'Achla03', 'Achla04', 'Achla06', 'Achle02', 'Achli01', 'Achli02', 'Achli03', 'Achlu02', 'Achma01', 'Achmi03', 'Achna02', 'Achne01', 'Achno01', 'Achpa01', 'Achpe02', 'Achpf01', 'Achps04', 'Achpu01', 'Achpy01', 'Achre01', 'Achro01', 'Achro02', 'Achru01', 'Achsa01', 'Achse02', 'Achsi01', 'Achst01', 'Achst03', 'Achsu06', 'Achsu07', 'Achte01', 'Achtr02', 'Achtr03', 'Achzh01', 'Achzi01', 'Actde01', 'Actno01', 'Adlba01', 'Adlbr01', 'Adlbr02', 'Adlmu02', 'Adlpa01', 'Adlsu01', 'Ampat01', 'Ampei01

In [43]:
pd.set_option('display.max_columns', None)

In [44]:
cleandf

,Achca02,Achla02,Achmi02,Achri01,Achsu03,Acidification_Status180D,Acidification_Status1Y,Acidification_Status90D,Amppe02,Aulam01,Coceu01,Cocli01,Cocpl01,CodeSite_SamplingOperations_y,Cycme02,Cycps01,Cymmi01,Cymsi01,Exiva01,Fragr01,Frave01,Gomex01,Gommi05,Gompa06,Gompu02,Mean180Days_1092,Mean180Days_1101,Mean180Days_1102,Mean180Days_1105,Mean180Days_1107,Mean180Days_1114,Mean180Days_1119,Mean180Days_1129,Mean180Days_1135,Mean180Days_1136,Mean180Days_1139,Mean180Days_1141,Mean180Days_1149,Mean180Days_1168,Mean180Days_1169,Mean180Days_1176,Mean180Days_1177,Mean180Days_1184,Mean180Days_1189,Mean180Days_1191,Mean180Days_1194,Mean180Days_1206,Mean180Days_1208,Mean180Days_1209,Mean180Days_1212,Mean180Days_1221,Mean180Days_1231,Mean180Days_1234,Mean180Days_1235,Mean180Days_1253,Mean180Days_1283,Mean180Days_1291,Mean180Days_1295,Mean180Days_1302,Mean180Days_1305,Mean180Days_1311,Mean180Days_1312,Mean180Days_1313,Mean180Days_1319,Mean180Days_1335,Mean180Days_1339,Mean180Days_1340,Mean180Days_1345,Mean180Days_1350,Mean180Days_1359,Mean180Days_1369,Mean180Days_1382,Mean180Days_1383,Mean180Days_1386,Mean180Days_1388,Mean180Days_1389,Mean180Days_1392,Mean180Days_1433,Mean180Days_1480,Mean180Days_1506,Mean180Days_1517,Mean180Days_1586,Mean180Days_1629,Mean180Days_1630,Mean180Days_1652,Mean180Days_1666,Mean180Days_1688,Mean180Days_1694,Mean180Days_1700,Mean180Days_1841,Mean180Days_1955,Mean180Days_1958,Mean180Days_1959,Mean1Y_1101,Mean1Y_1102,Mean1Y_1105,Mean1Y_1107,Mean1Y_1108,Mean1Y_1114,Mean1Y_1119,Mean1Y_1128,Mean1Y_1129,Mean1Y_1135,Mean1Y_1136,Mean1Y_1139,Mean1Y_1141,Mean1Y_1149,Mean1Y_1168,Mean1Y_1169,Mean1Y_1176,Mean1Y_1177,Mean1Y_1184,Mean1Y_1189,Mean1Y_1191,Mean1Y_1192,Mean1Y_1194,Mean1Y_1206,Mean1Y_1208,Mean1Y_1209,Mean1Y_1212,Mean1Y_1218,Mean1Y_1221,Mean1Y_1231,Mean1Y_1234,Mean1Y_1235,Mean1Y_1253,Mean1Y_1278,Mean1Y_1283,Mean1Y_1291,Mean1Y_1292,Mean1Y_1295,Mean1Y_1302,Mean1Y_1305,Mean1Y_1311,Mean1Y_1312,Mean1Y_1313,Mean1Y_1319,Mean1Y_1335,Mean1Y_1339,Mean1Y_1340,Mean1Y_1345,Mean1Y_1350,Mean1Y_1359,Mean1Y_1369,Mean1Y_1382,Mean1Y_1383,Mean1Y_1386,Mean1Y_1387,Mean1Y_1388,Mean1Y_1389,Mean1Y_1392,Mean1Y_1433,Mean1Y_1468,Mean1Y_1469,Mean1Y_1470,Mean1Y_1473,Mean1Y_1480,Mean1Y_1506,Mean1Y_1517,Mean1Y_1548,Mean1Y_1549,Mean1Y_1586,Mean1Y_1591,Mean1Y_1592,Mean1Y_1593,Mean1Y_1623,Mean1Y_1629,Mean1Y_1630,Mean1Y_1652,Mean1Y_1666,Mean1Y_1669,Mean1Y_1688,Mean1Y_1694,Mean1Y_1700,Mean1Y_1841,Mean1Y_1911,Mean1Y_1955,Mean1Y_1958,Mean1Y_1959,Mean1Y_6372,Mean1Y_XOMP,Mean90Days_1101,Mean90Days_1102,Mean90Days_1105,Mean90Days_1107,Mean90Days_1108,Mean90Days_1114,Mean90Days_1119,Mean90Days_1129,Mean90Days_1135,Mean90Days_1136,Mean90Days_1139,Mean90Days_1141,Mean90Days_1149,Mean90Days_1168,Mean90Days_1169,Mean90Days_1176,Mean90Days_1177,Mean90Days_1184,Mean90Days_1189,Mean90Days_1191,Mean90Days_1194,Mean90Days_1206,Mean90Days_1208,Mean90Days_1212,Mean90Days_1221,Mean90Days_1234,Mean90Days_1235,Mean90Days_1253,Mean90Days_1283,Mean90Days_1295,Mean90Days_1302,Mean90Days_1305,Mean90Days_1311,Mean90Days_1312,Mean90Days_1313,Mean90Days_1319,Mean90Days_1335,Mean90Days_1339,Mean90Days_1340,Mean90Days_1350,Mean90Days_1359,Mean90Days_1369,Mean90Days_1382,Mean90Days_1383,Mean90Days_1386,Mean90Days_1388,Mean90Days_1389,Mean90Days_1392,Mean90Days_1433,Mean90Days_1480,Mean90Days_1506,Mean90Days_1517,Mean90Days_1586,Mean90Days_1629,Mean90Days_1630,Mean90Days_1652,Mean90Days_1666,Mean90Days_1688,Mean90Days_1694,Mean90Days_1700,Mean90Days_1841,Mean90Days_1955,Mean90Days_1958,Mean90Days_1959,Melva01,MineralMicropollutants_Status180D,MineralMicropollutants_Status1Y,MineralMicropollutants_Status90D,Navan05,Navcr05,Navcr09,Navge02,Navgr01,Navla04,Navpe05,Navre04,Navrh02,Navsa03,Navsu03,Nitdi04,Nitfo01,Nitpa01,Nitrates_Status180D,Nitrates_Status1Y,Nitrates_Status90D,Nitrogencompounds_Status180D,Nitrogencompounds_Status1Y,Nitrogencompounds_Status90D,Nitso01,Nitso04,OrganicMatter_Status180D,OrganicMicropollutants_Status180D,OrganicMicropollutants_Status1Y,OrganicMicropollutants_Sta

In [37]:
cleandf.isna().mean()

TotalAbundance_SamplingOperation    0.000000
Achat02                             0.944048
Achca02                             0.762674
Achda01                             0.920391
Achda02                             0.901239
                                      ...   
Mean90Days_XOMP                     0.862561
Uncommon_Taxons                     0.000000
IBD                                 0.124296
IBD_EQR                             0.124296
IBD_EQR_Status                      0.124296
Length: 529, dtype: float64

In [45]:
nan_percentage = cleandf.isna().sum().sum() / cleandf.size * 100
print(f"{nan_percentage:.2f}% del DataFrame son NaNs.")


55.46% del DataFrame son NaNs.


In [46]:
import os

# Definir el path base
base_path = r"dfs_taxon_p_epm"

# Crear la carpeta si no existe
os.makedirs(base_path, exist_ok=True)

# Loop por regiones y guardar cada df
for region in regiones:
    cleandf = build_region_train_test_epm(
        region=region,
        sites=sites,
        taxones=taxones,
        pressure=pressure,
        train=train,
        test_codes=test_codes,
        fillna_with=0
    )

    # Nombre del archivo, por ejemplo df_PACA.parquet
    file_name = f"df_{region}.parquet"
    file_path = os.path.join(base_path, file_name)

    # Guardar el DataFrame
    save_with_key(cleandf, file_path + file_name)
    print(f"Guardado: {file_path}")


Dropped exact duplicate columns: ['Gompr03', 'Navkr01', 'Schex01', 'Mean180Days_1630', 'Mean180Days_1647', 'Mean180Days_1648', 'Mean1Y_1630', 'Mean1Y_1647', 'Mean1Y_1648', 'Mean1Y_2539', 'Mean1Y_6600', 'Mean90Days_1211', 'Mean90Days_1630', 'Mean90Days_1647', 'Mean90Days_1648', 'Mean90Days_1780', 'Mean90Days_1811', 'Mean90Days_2539', 'Mean90Days_2872', 'Mean90Days_5986']
Dropped columns with >95% missing: ['Achaf01', 'Achaf02', 'Achat01', 'Achca02', 'Achca04', 'Achch01', 'Achco01', 'Achda01', 'Achda02', 'Achde02', 'Achdr01', 'Achex01', 'Achex02', 'Achfu01', 'Achgr02', 'Achgr03', 'Achhi01', 'Achho01', 'Achho03', 'Achhu01', 'Achja02', 'Achko01', 'Achkr02', 'Achla03', 'Achla06', 'Achli03', 'Achob01', 'Achpf01', 'Achpu01', 'Achre01', 'Achro02', 'Achru01', 'Achth01', 'Achtr03', 'Achzh01', 'Adlbr02', 'Adlmi01', 'Adlsu01', 'Ampco01', 'Ampma01', 'Ampme01', 'Ampmi02', 'Ampmo01', 'Ampne01', 'Ampne02', 'Ampno01', 'Amppe01', 'Ampve01', 'Ampve02', 'Astfo01', 'Aulbr01', 'Auldi01', 'Aulps01', 'Aulsu01

In [47]:
def build_region_train_test_epm_no_clean(region, sites, taxones, pressure, train, test_codes, fillna_with=0):
    """
    Flujo completo por región:
    1) Filtra sites y epm (df, df1)
    1.5) Filtra sites y taxones (df, dfe)
    2) Pivotea abundancias permil por TaxonCode (wide)
    3) Une con pressure (df2) usando las mismas llaves que tú pusiste
    4) Saca train_df (inner contra 'train' por SamplingOperations_code)
    5) Saca test_df filtrando df2 por test_codes

    Parámetros:
      - region: int/str con el código de región (HERlvl1Code)
      - sites, taxones, pressure, train: DataFrames existentes
      - test_codes: iterable de SamplingOperations_code para test
      - fillna_with: si quieres rellenar NaN post-pivote (0 por default). Usa None para no rellenar.

    Regresa:
      - train_df, test_df
    """
    # 1) df y df1
    df1 = get_region_sites_and_taxa(region, sites, taxones)
    
    dfe = get_region_sites_and_epm(region, sites, epm)

    # Si no hay datos para la región, devuelve vacíos con mismas columnas.
    if df1.empty:
        empty_train = pd.DataFrame(columns=['SamplingOperations_code'])
        empty_test  = pd.DataFrame(columns=['SamplingOperations_code'])
        return empty_train, empty_test

    # 2) pivote
    wide = pivot_taxa_abundance_pm(df1)
    wide2=pivot_pressure_means(dfe)

    # 3) merge con pressure (exactamente tus llaves)
    df2 = pd.merge(
        wide,
        pressure,
        on=['SamplingOperations_code', 'CodeSite_SamplingOperations', 'Date_SamplingOperation'],
        how='inner'
    )

    df2 = pd.merge(
        df2,
        wide2,
        on=['SamplingOperations_code'],
        how='inner'
    )

   
    df3 = pd.merge(df2, train, on='SamplingOperations_code', how='left')    
    # 5) test_df (filtrado por test_codes)
    col = 'IBD'   # columna en dfB que checas

    mask = df3[col].isna()          # True si NaN

    train_df = df3.loc[~mask]
    test_df = df3.loc[mask]  
    test_df=test_df.drop(columns=(['IBD', 'IBD_EQR', 'IBD_EQR_Status']))

    cleandf = clean_up(train= train_df,test=test_df)

    return cleandf

In [48]:
import os

# Definir el path base
base_path = r"dfs_taxon_p_epm_dirty_af"

# Crear la carpeta si no existe
os.makedirs(base_path, exist_ok=True)

# Loop por regiones y guardar cada df
for region in regiones:
    cleandf = build_region_train_test_epm_no_clean(
        region=region,
        sites=sites,
        taxones=taxones,
        pressure=pressure,
        train=train,
        test_codes=test_codes,
        fillna_with=0
    )

    # Nombre del archivo, por ejemplo df_PACA.parquet
    file_name = f"df_{region}.parquet"
    file_path = os.path.join(base_path, file_name)

    # Guardar el DataFrame
    save_with_key(cleandf, file_path + file_name)
    print(f"Guardado: {file_path}")

Dropped exact duplicate columns: ['Gompr03', 'Navkr01', 'Schex01', 'Mean180Days_1630', 'Mean180Days_1647', 'Mean180Days_1648', 'Mean1Y_1630', 'Mean1Y_1647', 'Mean1Y_1648', 'Mean1Y_2539', 'Mean1Y_6600', 'Mean90Days_1211', 'Mean90Days_1630', 'Mean90Days_1647', 'Mean90Days_1648', 'Mean90Days_1780', 'Mean90Days_1811', 'Mean90Days_2539', 'Mean90Days_2872', 'Mean90Days_5986']
Dropped columns with >95% missing: ['Achaf01', 'Achaf02', 'Achat01', 'Achca02', 'Achca04', 'Achch01', 'Achco01', 'Achda01', 'Achda02', 'Achde02', 'Achdr01', 'Achex01', 'Achex02', 'Achfu01', 'Achgr02', 'Achgr03', 'Achhi01', 'Achho01', 'Achho03', 'Achhu01', 'Achja02', 'Achko01', 'Achkr02', 'Achla03', 'Achla06', 'Achli03', 'Achob01', 'Achpf01', 'Achpu01', 'Achre01', 'Achro02', 'Achru01', 'Achth01', 'Achtr03', 'Achzh01', 'Adlbr02', 'Adlmi01', 'Adlsu01', 'Ampco01', 'Ampma01', 'Ampme01', 'Ampmi02', 'Ampmo01', 'Ampne01', 'Ampne02', 'Ampno01', 'Amppe01', 'Ampve01', 'Ampve02', 'Astfo01', 'Aulbr01', 'Auldi01', 'Aulps01', 'Aulsu01

In [ ]:
def build_region_train_test_full( sites, taxones, pressure, train):
    """
    Flujo completo por región:
    1) Sites y taxones (df, df1)
    2) Pivotea abundancias permil por TaxonCode (wide)
    3) Une con pressure (df2) usando las mismas llaves que tú pusiste
    4) Saca train_df (inner contra 'train' por SamplingOperations_code)
    5) Saca test_df filtrando df2 por test_codes

    Parámetros:
      - region: int/str con el código de región (HERlvl1Code)
      - sites, taxones, pressure, train: DataFrames existentes
      - test_codes: iterable de SamplingOperations_code para test
      - fillna_with: si quieres rellenar NaN post-pivote (0 por default). Usa None para no rellenar.

    Regresa:
      - train_df, test_df
    """
    # 1) df y df1
    df1 = taxones

    # Si no hay datos para la región, devuelve vacíos con mismas columnas.
    if df1.empty:
        empty_train = pd.DataFrame(columns=['SamplingOperations_code'])
        empty_test  = pd.DataFrame(columns=['SamplingOperations_code'])
        return empty_train, empty_test

    # 2) pivote
    wide = pivot_taxa_abundance_pm(df1)

    # 3) merge con pressure (exactamente tus llaves)
    df2 = pd.merge(
        wide,
        pressure,
        on=['SamplingOperations_code', 'CodeSite_SamplingOperations', 'Date_SamplingOperation'],
        how='inner'
    )

    df3 = pd.merge(df2, sites, on='SamplingOperations_code', how='left')  

    df3 = pd.merge(df3, train, on='SamplingOperations_code', how='left')    
    # 5) test_df (filtrado por test_codes)
    col = 'IBD'   # columna en dfB que checas

    mask = df3[col].isna()          # True si NaN

    train_df = df3.loc[~mask]
    test_df = df3.loc[mask]  
    test_df=test_df.drop(columns=(['IBD', 'IBD_EQR', 'IBD_EQR_Status']))

    cleandf = clean_up(train= train_df,test=test_df)

    return cleandf

In [51]:
clean_taxones_pressure = build_region_train_test_full( sites, taxones, pressure, train)

Dropped exact duplicate columns: ['Cocps01', 'Dipde01', 'Dippa01', 'Encsu01', 'Encsu02', 'Ethpu01', 'Grama01', 'Navpa06', 'Navth01', 'Navtr01', 'Nitth01', 'Parhe01', 'Plata01', 'Semro01', 'Synaf01', 'Thani01', 'CodeSite_SamplingOperations_y']
Dropped columns with >95% missing: ['Achaa01', 'Achac01', 'Achaf01', 'Achaf02', 'Achal01', 'Acham01', 'Achan01', 'Achat01', 'Achat03', 'Achba01', 'Achbi01', 'Achbi02', 'Achbr01', 'Achca01', 'Achca03', 'Achca04', 'Achch01', 'Achcl01', 'Achco01', 'Achco03', 'Achco04', 'Achcr01', 'Achcy01', 'Achda01', 'Achda02', 'Achde01', 'Achde02', 'Achdi01', 'Achdi02', 'Achel01', 'Achen01', 'Achex01', 'Achex02', 'Achex03', 'Achfl01', 'Achfr01', 'Achfu01', 'Achgr01', 'Achgr02', 'Achgr03', 'Achha01', 'Achhe01', 'Achhi01', 'Achho01', 'Achho02', 'Achho03', 'Achhu01', 'Achim01', 'Achim02', 'Achim03', 'Achin01', 'Achin02', 'Achja01', 'Achja02', 'Achjo01', 'Achko01', 'Achkr01', 'Achkr02', 'Achkr03', 'Achkr04', 'Achku01', 'Achla01', 'Achla03', 'Achla05', 'Achla06', 'Achle